# 结构优先与长度兜底

结构切分和长度切分不是二选一。常见生产策略是先按照标题、Section 或 Element 得到语义完整的块，再只对超长块进行二次切分。

```text
Markdown 文档
    ↓ MarkdownHeaderTextSplitter
章节 Document + 标题 metadata
    ↓ RecursiveCharacterTextSplitter
满足长度限制的最终 Chunk，继续保留标题 metadata
```

In [ ]:
from pathlib import Path
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

CURRENT_DIR = Path.cwd().resolve()
RAG_DIR = next(
    (path for path in (CURRENT_DIR, *CURRENT_DIR.parents) if path.name == "6-LangChain中的RAG"),
    CURRENT_DIR / "系统学习" / "6-LangChain中的RAG",
)
path = RAG_DIR / "asset" / "load" / "11-langchain.md"
markdown = path.read_text(encoding="utf-8")

structure_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "chapter"),
        ("##", "section"),
        ("###", "subsection"),
    ],
    strip_headers=False,
)
section_documents = structure_splitter.split_text(markdown)

length_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    add_start_index=True,
)
final_chunks = length_splitter.split_documents(section_documents)


In [ ]:
print(f"结构章节数：{len(section_documents)}")
print(f"最终 Chunk 数：{len(final_chunks)}")
for chunk in final_chunks[:5]:
    print(chunk.metadata)
    print(chunk.page_content[:180])
    print("-" * 50)


这种组合保留了章节层级，并防止单个章节无限增长。对于 PDF 或 DOCX，可以把第一阶段替换为 Docling/MinerU 提供的 Section/Element；第二阶段仍使用目标 Tokenizer 做上限控制。